# Turkish WikiNER Named Entity Recognition with Qwen3-0.6B

This notebook explores Turkish WikiNER, prepares a focused NER dataset, fine-tunes Qwen3-0.6B with LoRA, and evaluates a saved checkpoint.

- Dataset: [turkish-nlp-suite/turkish-wikiNER](https://huggingface.co/datasets/turkish-nlp-suite/turkish-wikiNER)
- Base model: [Qwen/Qwen3-0.6B](https://huggingface.co/Qwen/Qwen3-0.6B)
- Target labels: `PERSON`, `GPE`, `DATE`, `ORG`, `NORP`, `TITLE`, and `EVENT`
- The base model is loaded in FP16 without quantization; LoRA adapters are trained.

The reported benchmark uses the saved `checkpoint-733` adapter selected for this experiment.
Stored cell outputs are snapshots from the recorded run; this cleanup did not rerun training or inference.


Repository note: aggregate preparation statistics and benchmark metrics are retained. Saved outputs that expose individual dataset records or model examples have been removed. The remaining recorded outputs are snapshots; training and inference were not rerun for this cleanup.

## 1. Dataset inspection

In [ ]:
from datasets import load_dataset
from collections import Counter
from statistics import median
from IPython.display import display
import pandas as pd
import random

# Load the dataset from Hugging Face.
raw_data = load_dataset("turkish-nlp-suite/turkish-wikiNER")
data = raw_data  # Read-only alias for the initial source-data inspection.

print(data)
first_split = next(iter(data))
print("Columns:", data[first_split].column_names)
print("Example row:", data[first_split][0])

# Use label names from the dataset features when available.
# If tags are numeric, decode them using the repository's label order.
entity_types = [
    "CARDINAL", "DATE", "EVENT", "FAC", "GPE", "LANGUAGE", "LAW",
    "LOC", "MONEY", "NORP", "ORDINAL", "ORG", "PERCENT", "PERSON",
    "PRODUCT", "QUANTITY", "TIME", "TITLE", "WORK_OF_ART"
]

fallback_labels = {0: "O"}
for i, entity_type in enumerate(entity_types, start=1):
    fallback_labels[i] = f"B-{entity_type}"
    fallback_labels[i + len(entity_types)] = f"I-{entity_type}"

tag_feature = data[first_split].features["tags"]
while hasattr(tag_feature, "feature") and not hasattr(tag_feature, "names"):
    tag_feature = tag_feature.feature

label_names = getattr(tag_feature, "names", None)

def decode_tag(tag):
    if isinstance(tag, str) and not tag.strip().lstrip("-").isdigit():
        return tag

    try:
        tag_id = int(tag)
    except (TypeError, ValueError):
        return str(tag)

    if label_names and 0 <= tag_id < len(label_names):
        return label_names[tag_id]

    return fallback_labels.get(tag_id, f"ID_{tag_id}")

def extract_entities(tokens, tags):
    """Extract (entity type, text) pairs from BIO tags."""
    found = []
    current_type = None
    current_tokens = []

    for token, tag in zip(tokens, tags):
        if tag.startswith("B-"):
            if current_tokens:
                found.append((current_type, " ".join(current_tokens)))
            current_type = tag[2:]
            current_tokens = [token]

        elif tag.startswith("I-"):
            entity_type = tag[2:]
            if current_type == entity_type:
                current_tokens.append(token)
            else:
                if current_tokens:
                    found.append((current_type, " ".join(current_tokens)))
                current_type = entity_type
                current_tokens = [token]

        else:
            if current_tokens:
                found.append((current_type, " ".join(current_tokens)))
            current_type = None
            current_tokens = []

    if current_tokens:
        found.append((current_type, " ".join(current_tokens)))

    return found

# Basic statistics and label distribution for each split.
summary_rows = []
entity_rows = []
train_name = "train" if "train" in data else first_split
example_indices = []

for split_name, split in data.items():
    lengths = []
    token_tag_mismatches = 0
    orphan_i_tags = 0
    sentences_without_entities = 0
    total_entities = 0
    entity_counts = Counter()

    for row_index, row in enumerate(split):
        tokens = row["tokens"]
        tags = [decode_tag(tag) for tag in row["tags"]]
        lengths.append(len(tokens))

        if len(tokens) != len(tags):
            token_tag_mismatches += 1

        if not any(tag.startswith("B-") for tag in tags):
            sentences_without_entities += 1

        previous_type = None
        for tag in tags:
            if tag.startswith("B-"):
                entity_type = tag[2:]
                entity_counts[entity_type] += 1
                total_entities += 1
                previous_type = entity_type

            elif tag.startswith("I-"):
                entity_type = tag[2:]
                if previous_type != entity_type:
                    orphan_i_tags += 1
                previous_type = entity_type

            else:
                previous_type = None

        if split_name == train_name and any(tag.startswith("B-") for tag in tags):
            example_indices.append(row_index)

    summary_rows.append({
        "sentence": len(split),
        "total tokens": sum(lengths),
        "mean tokens/sentence": round(sum(lengths) / len(lengths), 1) if lengths else 0,
        "median tokens/sentence": median(lengths) if lengths else 0,
        "entity count": total_entities,
        "sentences without entities (%)": round(100 * sentences_without_entities / len(split), 1) if len(split) else 0,
        "token-label mismatches": token_tag_mismatches,
        "orphan I-tags": orphan_i_tags,
    })

    for entity_type, count in entity_counts.items():
        entity_rows.append({
            "entity type": entity_type,
            "split": split_name,
            "count": count,
        })

print("\nSplit summary:")
display(pd.DataFrame(summary_rows, index=list(data.keys())).rename_axis("split"))

if entity_rows:
    entity_table = (
        pd.DataFrame(entity_rows)
        .pivot_table(index="entity type", columns="split", values="count", aggfunc="sum", fill_value=0)
    )
    entity_table["total"] = entity_table.sum(axis=1)
    print("\nCounts by entity type:")
    display(entity_table.sort_values("total", ascending=False))

# Show five labeled examples from the training split using a fixed seed.
print("\nLabeled examples:")
sample_indices = random.Random(42).sample(
    example_indices, k=min(5, len(example_indices))
)

for row_index in sample_indices:
    row = data[train_name][row_index]
    tags = [decode_tag(tag) for tag in row["tags"]]
    entities = extract_entities(row["tokens"], tags)

    print(f"\nSentence: {' '.join(row['tokens'])}")
    print("Entities:", "; ".join(
        f"{entity_type}: {text}" for entity_type, text in entities
    ))

,sentence,total tokens,mean tokens/sentence,median tokens/sentence,entity count,sentences without entities (%),token-label mismatches,orphan I-tags
split,,,,,,,,
train,17967,303443,16.9,16.0,51764,10.6,0,0
validation,1000,16658,16.7,15.0,2969,11.7,0,0
test,1000,17275,17.3,16.0,3016,10.4,0,0


split,test,train,validation,total
entity type,,,,
PERSON,654,11492,641,12787
GPE,566,9209,593,10368
DATE,343,6245,335,6923
ORG,250,4081,252,4583
CARDINAL,240,3876,179,4295
NORP,212,3636,175,4023
WORK_OF_ART,158,2637,156,2951
TITLE,153,2136,205,2494
EVENT,95,2197,100,2392


In [ ]:
# Exclude the first five examples so they are not shown again.
first_examples = set(
    random.Random(42).sample(example_indices, k=min(5, len(example_indices)))
)
remaining_examples = [i for i in example_indices if i not in first_examples]

additional_examples = random.Random(43).sample(
    remaining_examples, k=min(25, len(remaining_examples))
)

print("\n25 additional labeled examples (if available):")
for row_index in additional_examples:
    row = data[train_name][row_index]
    tags = [decode_tag(tag) for tag in row["tags"]]
    entities = extract_entities(row["tokens"], tags)

    print(f"\nSentence: {' '.join(row['tokens'])}")
    print("Entities:", "; ".join(
        f"{entity_type}: {text}" for entity_type, text in entities
    ))

## 2. Target-label exploration

In [ ]:
from datasets import DatasetDict

target_types = {"PERSON", "GPE", "DATE", "ORG", "NORP", "TITLE", "EVENT"}
original_decode_tag = decode_tag  # Keep the original decoder for source labels.

# O plus the B- and I- labels for the selected entity types.
label2id = {"O": 0}
for entity_type in ["PERSON", "GPE", "DATE", "ORG", "NORP", "TITLE", "EVENT"]:
    for prefix in ["B", "I"]:
        label2id[f"{prefix}-{entity_type}"] = len(label2id)

id2label = {v: k for k, v in label2id.items()}

def is_target_label(label):
    parts = label.split("-", 1)
    return (
        len(parts) == 2
        and parts[0] in {"B", "I"}
        and parts[1].upper() in target_types
    )

def convert_labels(example):
    original_labels = [original_decode_tag(tag) for tag in example["tags"]]
    has_target_entity = any(is_target_label(tag) for tag in original_labels)

    new_labels = [
        label2id[tag] if is_target_label(tag) else label2id["O"]
        for tag in original_labels
    ]

    return {
        "_has_target_entity": has_target_entity,
        "_new_tags": new_labels,
    }

filtered_splits = {}

for split_name, split in raw_data.items():
    temporary = split.map(convert_labels)
    temporary = temporary.filter(lambda row: row["_has_target_entity"])
    temporary = temporary.remove_columns(["tags", "_has_target_entity"])
    temporary = temporary.rename_column("_new_tags", "tags")
    filtered_splits[split_name] = temporary

    print(f"{split_name}: {len(split):,} → {len(temporary):,} rows")

# Use the filtered data for subsequent analysis and training.
analysis_data = DatasetDict(filtered_splits)

# Decode the new 15 classes using their correct names.
def decode_tag(tag):
    if isinstance(tag, str) and not tag.strip().lstrip("-").isdigit():
        return tag
    return id2label.get(int(tag), f"ID_{tag}")

print("\nNew label mapping:", id2label)
print(analysis_data)

train: 17,967 → 14,446 rows
validation: 1,000 → 799 rows
test: 1,000 → 813 rows

New label mapping: {0: 'O', 1: 'B-PERSON', 2: 'I-PERSON', 3: 'B-GPE', 4: 'I-GPE', 5: 'B-DATE', 6: 'I-DATE', 7: 'B-ORG', 8: 'I-ORG', 9: 'B-NORP', 10: 'I-NORP', 11: 'B-TITLE', 12: 'I-TITLE', 13: 'B-EVENT', 14: 'I-EVENT'}
DatasetDict({
    train: Dataset({
        features: ['tokens', 'tags'],
        num_rows: 14446
    })
    validation: Dataset({
        features: ['tokens', 'tags'],
        num_rows: 799
    })
    test: Dataset({
        features: ['tokens', 'tags'],
        num_rows: 813
    })
})


In [ ]:
import random

def extract_entities(tokens, tags):
    found_entities = []
    current_type = None
    current_tokens = []

    def flush_entity():
        if current_tokens:
            found_entities.append((current_type, " ".join(current_tokens)))

    for token, tag_id in zip(tokens, tags):
        tag = decode_tag(tag_id)

        if tag.startswith("B-"):
            flush_entity()
            current_type = tag[2:]
            current_tokens = [token]

        elif tag.startswith("I-"):
            entity_type = tag[2:]
            if current_type == entity_type:
                current_tokens.append(token)
            else:
                flush_entity()
                current_type = entity_type
                current_tokens = [token]

        else:
            flush_entity()
            current_type = None
            current_tokens = []

    flush_entity()
    return found_entities


train = analysis_data["train"]
sample_indices = random.Random(42).sample(
    range(len(train)),
    k=min(5, len(train))
)

for sequence_number, index in enumerate(sample_indices, start=1):
    example = train[index]
    tokens = example["tokens"]
    tags = [decode_tag(tag) for tag in example["tags"]]
    entities = extract_entities(tokens, example["tags"])

    output = "; ".join(
        f"{entity_type}: {text}" for entity_type, text in entities
    )

    print(f"\n--- Example {sequence_number} ---")
    print("Sentence:", " ".join(tokens))
    print("Remaining labels:", [
        f"{token} → {tag}"
        for token, tag in zip(tokens, tags)
        if tag != "O"
    ])
    print("Target output:", f"Varlıklar: {output}" if output else "Varlıklar: Yok")

In [ ]:
import pandas as pd

summary = []

for split_name, split in analysis_data.items():
    lengths = pd.Series([len(example["tokens"]) for example in split])

    summary.append({
        "Split": split_name,
        "Sentence count": len(lengths),
        "Mean tokens/sentence": round(lengths.mean(), 1),
        "25th percentile": round(lengths.quantile(0.25), 1),
        "Median (50th percentile)": round(lengths.quantile(0.50), 1),
        "75th percentile": round(lengths.quantile(0.75), 1),
        "90th percentile": round(lengths.quantile(0.90), 1),
        "Shortest": int(lengths.min()),
        "Longest": int(lengths.max()),
    })

display(pd.DataFrame(summary).set_index("Split"))

,Sentence count,Mean tokens/sentence,25th percentile,Median (50th percentile),75th percentile,90th percentile,Shortest,Longest
Split,,,,,,,,
train,14446,17.2,11.0,16.0,22.0,28.0,3,50
validation,799,16.9,11.0,16.0,22.0,27.0,3,49
test,813,17.5,11.0,17.0,22.0,28.0,3,44


## 3. Training-data preparation

Labels outside the seven target types are mapped to `O`. To reduce dominant combinations, 68% of training sentences containing only `PERSON`, only `GPE`, or exactly `PERSON + GPE` are randomly removed (seed 42). The final training split keeps the remaining labeled examples and samples empty-target examples to make up about 5% of training data. Validation and test splits are unchanged.


In [ ]:
from datasets import DatasetDict

# Reuse raw_data loaded in the first section.
target_types = ["PERSON", "GPE", "DATE", "ORG", "NORP", "TITLE", "EVENT"]

# New label mapping: O plus B/I labels for the target entity types.
label2id = {"O": 0}
for entity_type in target_types:
    for prefix in ["B", "I"]:
        label2id[f"{prefix}-{entity_type}"] = len(label2id)

id2label = {label_id: label for label, label_id in label2id.items()}

# Get raw label names from the dataset feature metadata.
first_split = next(iter(raw_data))
tag_feature = raw_data[first_split].features["tags"]

while hasattr(tag_feature, "feature") and not hasattr(tag_feature, "names"):
    tag_feature = tag_feature.feature

raw_label_names = getattr(tag_feature, "names", None)

# If feature names are unavailable, use the numeric label order from HF.
raw_entity_types = [
    "CARDINAL", "DATE", "EVENT", "FAC", "GPE", "LANGUAGE", "LAW",
    "LOC", "MONEY", "NORP", "ORDINAL", "ORG", "PERCENT", "PERSON",
    "PRODUCT", "QUANTITY", "TIME", "TITLE", "WORK_OF_ART"
]

raw_id2label = {0: "O"}
for i, entity_type in enumerate(raw_entity_types, start=1):
    raw_id2label[i] = f"B-{entity_type}"
    raw_id2label[i + len(raw_entity_types)] = f"I-{entity_type}"

def decode_raw_label(label):
    if isinstance(label, str) and not label.strip().lstrip("-").isdigit():
        return label.upper()

    label_id = int(label)

    if raw_label_names and 0 <= label_id < len(raw_label_names):
        return raw_label_names[label_id].upper()

    return raw_id2label[label_id]

def map_target_labels(example):
    new_labels = []

    for original_label in example["tags"]:
        label = decode_raw_label(original_label)

        # Keep the seven target types; map all other labels to O.
        new_labels.append(label2id.get(label, 0))

    return {
        "_new_tags": new_labels,
        "_has_target_entity": any(label != 0 for label in new_labels),
    }

prepared_splits = {}
target_counts = {}

for split_name, split in raw_data.items():
    temporary = split.map(
        map_target_labels,
        desc=f"{split_name} labels are being mapped",
    )

    positive_count = sum(temporary["_has_target_entity"])
    target_counts[split_name] = positive_count

    # Keep rows without target entities; their labels are set to O.
    temporary = temporary.remove_columns(["tags", "_has_target_entity"])
    temporary = temporary.rename_column("_new_tags", "tags")
    prepared_splits[split_name] = temporary

training_data = DatasetDict(prepared_splits)

for split_name, split in training_data.items():
    positive_count = target_counts[split_name]
    negative_count = len(split) - positive_count
    print(
        f"{split_name}: total {len(split):,} | "
        f"sentences with target entities {positive_count:,} | "
        f"sentences without target entities {negative_count:,}"
    )

print("\nNew label mapping:", id2label)


train: total 17,967 | sentences with target entities 14,446 | sentences without target entities 3,521
validation: total 1,000 | sentences with target entities 799 | sentences without target entities 201
test: total 1,000 | sentences with target entities 813 | sentences without target entities 187

New label mapping: {0: 'O', 1: 'B-PERSON', 2: 'I-PERSON', 3: 'B-GPE', 4: 'I-GPE', 5: 'B-DATE', 6: 'I-DATE', 7: 'B-ORG', 8: 'I-ORG', 9: 'B-NORP', 10: 'I-NORP', 11: 'B-TITLE', 12: 'I-TITLE', 13: 'B-EVENT', 14: 'I-EVENT'}


In [ ]:
def normalize_label(label):
    # If the label is numeric, look up its text in id2label.
    if isinstance(label, int):
        label = id2label.get(label, id2label.get(str(label), str(label)))

    label = str(label).upper()

    # Remove BIO prefixes: B-PERSON / I-PERSON -> PERSON.
    for prefix in ("B-", "I-", "B_", "I_"):
        if label.startswith(prefix):
            label = label[len(prefix):]
            break

    return label


def labels_in_sentence(labels):
    found_entities = set()

    for label in labels:
        label = normalize_label(label)

        if label != "O":
            found_entities.add(label)

    return found_entities


# Inspect each DatasetDict split separately, or a single Dataset as one split.
if hasattr(training_data, "items"):
    splits = list(training_data.items())
else:
    splits = [("training_data", training_data)]

print(f"{'Split':<12} {'PERSON only':>15} {'GPE only':>13} "
      f"{'Exactly PERSON + GPE':>25} {'Both present (may include others)':>34}")

for split_name, split_data in splits:
    person_only_count = 0
    gpe_only_count = 0
    exact_person_gpe_count = 0
    both_present_count = 0

    for example in split_data:
        labels = labels_in_sentence(example["tags"])

        if labels == {"PERSON"}:
            person_only_count += 1

        if labels == {"GPE"}:
            gpe_only_count += 1

        if labels == {"PERSON", "GPE"}:
            exact_person_gpe_count += 1

        if {"PERSON", "GPE"}.issubset(labels):
            both_present_count += 1

    print(f"{split_name:<12} {person_only_count:>15} {gpe_only_count:>13} "
          f"{exact_person_gpe_count:>25} {both_present_count:>34}")

Split            PERSON only      GPE only      Exactly PERSON + GPE  Both present (may include others)
train                   2498          1937                       443                               1678
validation               122           119                        25                                103
test                     138           125                        19                                 94


In [ ]:
import random

rng = random.Random(42)

categories = {
    "PERSON only": {"PERSON"},
    "GPE only": {"GPE"},
    "Exactly PERSON + GPE": {"PERSON", "GPE"},
}

train = training_data["train"]
category_indices = {name: [] for name in categories}

# Find each sentence's label set and add it to the matching category.
for i, example in enumerate(train):
    labels = labels_in_sentence(example["tags"])

    for category_name, target_labels in categories.items():
        if labels == target_labels:
            category_indices[category_name].append(i)
            break

indices_to_remove = set()

for category_name, indices in category_indices.items():
    removal_count = round(len(indices) * 0.68)
    indices_to_remove.update(rng.sample(indices, removal_count))

    print(
        f"{category_name}: "
        f"before={len(indices)}, "
        f"removed={removal_count}, "
        f"remaining={len(indices) - removal_count}"
    )

# Select the remaining training rows; leave validation and test unchanged.
retained_indices = [
    i for i in range(len(train))
    if i not in indices_to_remove
]

training_data["train"] = train.select(retained_indices)

print("\nUpdated split sizes:")
for split_name, split_data in training_data.items():
    print(f"{split_name}: {len(split_data)} sentence")

PERSON only: before=2498, removed=1699, remaining=799
GPE only: before=1937, removed=1317, remaining=620
Exactly PERSON + GPE: before=443, removed=301, remaining=142

Updated split sizes:
train: 14650 sentence
validation: 1000 sentence
test: 1000 sentence


In [ ]:
import random

train = training_data["train"]
positive_indices = []
empty_indices = []

for i, example in enumerate(train):
    if labels_in_sentence(example["tags"]):
        positive_indices.append(i)
    else:
        empty_indices.append(i)

# Keep empty-answer examples at about 5% of the final training set.
empty_keep_count = round(len(positive_indices) * 0.05 / 0.95)
retained_empty_indices = set(
    random.Random(42).sample(empty_indices, empty_keep_count)
)

retained_indices = sorted(
    positive_indices + list(retained_empty_indices)
)
training_data["train"] = train.select(retained_indices)

print(f"Labeled training sentences: {len(positive_indices):,}")
print(f"Initial empty training sentences: {len(empty_indices):,}")
print(f"Retained empty training sentences: {empty_keep_count:,}")
print(f"Updated training total: {len(training_data['train']):,}")
print("Validation/test unchanged:", len(training_data["validation"]), len(training_data["test"]))


Labeled training sentences: 11,129
Initial empty training sentences: 3,521
Retained empty training sentences: 586
Updated training total: 11,715
Validation/test unchanged: 1000 1000


In [ ]:
from collections import Counter
import pandas as pd

def get_label_name(tag):
    if isinstance(tag, str) and not tag.lstrip("-").isdigit():
        return tag
    return id2label[int(tag)]

target_entity_types = ["PERSON", "GPE", "DATE", "ORG", "NORP", "TITLE", "EVENT"]
entity_counts = Counter()
sentence_counts = Counter()

for split_name, split in training_data.items():
    for example in split:
        types_found = set()

        for tag_id in example["tags"]:
            tag = get_label_name(tag_id)

            # Each B- tag starts an entity; count entities from these tags.
            if tag.startswith("B-"):
                entity_type = tag[2:]
                entity_counts[(split_name, entity_type)] += 1
                types_found.add(entity_type)

        # Count a sentence once per type, even if that type occurs multiple times.
        for entity_type in types_found:
            sentence_counts[(split_name, entity_type)] += 1

rows = []

for split_name in training_data.keys():
    for entity_type in target_entity_types:
        rows.append({
            "Split": split_name,
            "Entity type": entity_type,
            "Entity count": entity_counts[(split_name, entity_type)],
            "Sentences containing this type": sentence_counts[(split_name, entity_type)],
        })

display(pd.DataFrame(rows))

,Split,Entity type,Entity count,Sentences containing this type
0,train,PERSON,8131,5336
1,train,GPE,6140,3918
2,train,DATE,6245,5096
3,train,ORG,4081,2939
4,train,NORP,3636,2784
5,train,TITLE,2136,1630
6,train,EVENT,2197,1755
7,validation,PERSON,641,406
8,validation,GPE,593,335
9,validation,DATE,335,273


## 4. Instruction-tuning examples

Ten Turkish command variants are used as equivalent requests for the same NER task. An instruction is attached to 80% of examples; the other 20% contain the sentence without an extra user instruction. The system message restricts extraction to the seven target labels. Empty-target examples use `Varlıklar: Yok`.


In [ ]:
from datasets import DatasetDict
import random

instructions = [
    "Özellik çıkar.",
    "Önemli bilgileri çıkar.",
    "Önemli kelimeleri çıkar.",
    "Anahtar kelimeleri çıkar.",
    "Metindeki varlıkları bul.",
    "Varlıkları türleriyle listele.",
    "Tanımlı etiketlerdeki varlıkları çıkar.",
    "Kişi, yer, tarih, kurum, grup, unvan ve olay adlarını tespit et.",
    "Cümlede geçen hedef varlıkları belirle.",
    "Varlıkları belirtilen kategorilere göre sınıflandır.",
]

system_message = (
    "Sen bir Named Entity Recognition (NER) modelisin. "
    "Verilen cümledeki varlıkları yalnızca PERSON, GPE, DATE, ORG, "
    "NORP, TITLE ve EVENT etiketlerini kullanarak çıkar. "
    'Yalnızca "Varlıklar: ETİKET: varlık; ..." biçiminde yanıt ver.'
)

def get_label_name(label):
    if isinstance(label, str) and not label.lstrip("-").isdigit():
        return label
    return id2label[int(label)]

def detokenize(tokens):
    attach_before = {".", ",", "!", "?", ";", ":", ")", "]", "}", "%", "'", "’"}
    attach_after = {"(", "[", "{", "“", "«"}

    text = ""
    previous_token = ""

    for token in tokens:
        if not text:
            text = token
        elif token in attach_before or previous_token in attach_after or previous_token in {"'", "’"}:
            text += token
        else:
            text += " " + token
        previous_token = token

    return text

def extract_entities(tokens, labels):
    found_entities = []
    current_type = None
    current_tokens = []

    def flush_entity():
        if current_tokens:
            found_entities.append((current_type, detokenize(current_tokens)))

    for token, label_id in zip(tokens, labels):
        label = get_label_name(label_id)

        if label.startswith("B-"):
            flush_entity()
            current_type = label[2:]
            current_tokens = [token]

        elif label.startswith("I-"):
            next_type = label[2:]
            if current_type == next_type:
                current_tokens.append(token)
            else:
                flush_entity()
                current_type = next_type
                current_tokens = [token]

        else:
            flush_entity()
            current_type = None
            current_tokens = []

    flush_entity()
    return found_entities

def build_instruction_mapping(row_count, split_name):
    indices = list(range(row_count))
    seed = 2026 + sum(ord(character) for character in split_name)
    random.Random(seed).shuffle(indices)

    prompted_count = round(row_count * 0.80)
    selected_indices = indices[:prompted_count]

    # Distribute instructions evenly across prompted examples.
    return {
        index: instructions[sequence_number % len(instructions)]
        for sequence_number, index in enumerate(selected_indices)
    }

def build_training_example(example, index, instruction_mapping):
    tokens = example["tokens"]
    entities = extract_entities(tokens, example["tags"])

    entity_text = "; ".join(
        f"{entity_type}: {expression}" for entity_type, expression in entities
    )
    assistant_message = (
        f"Varlıklar: {entity_text}" if entity_text else "Varlıklar: Yok"
    )

    sentence = detokenize(tokens)
    instruction = instruction_mapping.get(index)

    if instruction is None:
        user_message = f"Cümle: {sentence}"
    else:
        user_message = f"{instruction}\n\nCümle: {sentence}"

    return {
        "messages": [
            {"role": "system", "content": system_message},
            {"role": "user", "content": user_message},
            {"role": "assistant", "content": assistant_message},
        ]
    }

prepared_splits = {}
instruction_mappings = {}

for split_name, split in training_data.items():
    instruction_mapping = build_instruction_mapping(len(split), split_name)
    instruction_mappings[split_name] = instruction_mapping

    prepared_splits[split_name] = split.map(
        build_training_example,
        with_indices=True,
        fn_kwargs={"instruction_mapping": instruction_mapping},
        remove_columns=split.column_names,
        desc=f"Preparing {split_name} chat examples",
    )

sft_data = DatasetDict(prepared_splits)

for split_name, split in training_data.items():
    prompted_count = len(instruction_mappings[split_name])
    percentage = 100 * prompted_count / len(split) if len(split) else 0
    print(
        f"{split_name}: {len(split):,} examples | "
        f"{prompted_count:,} prompted (%{percentage:.1f}) | "
        f"{len(split) - prompted_count:,} unprompted"
    )

# Show one prompted and one unprompted training example.
train_examples = sft_data["train"]
train_instruction_mapping = instruction_mappings["train"]

sample_indices = []
prompted_index = next(iter(train_instruction_mapping), None)
unprompted_index = next(
    (i for i in range(len(train_examples)) if i not in train_instruction_mapping),
    None,
)

for index in [prompted_index, unprompted_index]:
    if index is not None:
        sample_indices.append(index)

for sequence_number, index in enumerate(sample_indices, start=1):
    print(f"\n--- Example {sequence_number} ---")
    for message in train_examples[index]["messages"]:
        print(f"\n{message['role'].upper()}:\n{message['content']}")

## 5. Prompt and tokenization previews

In [ ]:
import random

batch_size = 2
num_preview_batches = 3
num_preview_examples = batch_size * num_preview_batches
train_sft = sft_data["train"]

preview_indices = random.Random(42).sample(
    range(len(train_sft)),
    k=min(num_preview_examples, len(train_sft)),
)

preview_batches = [
    [train_sft[index] for index in preview_indices[start:start + batch_size]]
    for start in range(0, len(preview_indices), batch_size)
]

for batch_number, batch in enumerate(preview_batches, start=1):
    print(f"\n=== BATCH {batch_number} | {len(batch)} examples | before tokenization ===")

    for example_number, example in enumerate(batch, start=1):
        print(f"\n--- Example {example_number} ---")
        for message in example["messages"]:
            print(f"\n{message['role'].upper()}:\n{message['content']}")


In [ ]:
from transformers import AutoTokenizer

model_name = "Qwen/Qwen3-0.6B"
tokenizer = AutoTokenizer.from_pretrained(model_name)

for batch_number, batch in enumerate(preview_batches, start=1):
    print(f"\n=== TOKENIZED BATCH {batch_number} ===")

    for example_number, example in enumerate(batch, start=1):
        encoded = tokenizer.apply_chat_template(
            example["messages"],
            tokenize=True,
            add_generation_prompt=False,
            return_dict=True,
        )
        input_ids = encoded["input_ids"]

        print(f"\n--- Example {example_number} | {len(input_ids)} tokens ---")
        print("First 40 tokens:", tokenizer.convert_ids_to_tokens(input_ids[:40]))
        print("Decoded conversation with special tokens:")
        print(tokenizer.decode(input_ids, skip_special_tokens=False))


## 6. Sequence-length analysis

In [ ]:
import pandas as pd

candidate_max_lengths = [128, 192, 256, 384, 512]
length_rows = []

for split_name, split in sft_data.items():
    token_lengths = []

    for example in split:
        encoded = tokenizer.apply_chat_template(
            example["messages"],
            tokenize=True,
            add_generation_prompt=False,
            return_dict=True,
        )
        token_lengths.append(len(encoded["input_ids"]))

    lengths = pd.Series(token_lengths)
    row = {
        "Split": split_name,
        "Examples": len(lengths),
        "Mean tokens": round(lengths.mean(), 1),
        "Median": int(lengths.quantile(0.50)),
        "P90": int(lengths.quantile(0.90)),
        "P95": int(lengths.quantile(0.95)),
        "P99": int(lengths.quantile(0.99)),
        "Maximum": int(lengths.max()),
    }

    for max_length in candidate_max_lengths:
        row[f"Over {max_length}"] = int((lengths > max_length).sum())

    length_rows.append(row)

display(pd.DataFrame(length_rows).set_index("Split"))


,Examples,Mean tokens,Median,P90,P95,P99,Maximum,Over 128,Over 192,Over 256,Over 384,Over 512
Split,,,,,,,,,,,,
train,11715,164.3,160,205,220,252,459,10720,1938,88,1,0
validation,1000,157.2,151,195,216,247,289,855,116,7,0,0
test,1000,159.1,154,200,214,250,283,869,138,5,0,0


## 7. Runtime and model setup

The environment summary records the Python, library, CUDA, and GPU versions used for this run. `bitsandbytes` may appear in the installed-package inventory, but the model is loaded without quantization.


In [1]:
from pathlib import Path
import sys
import importlib.util
from importlib.metadata import version, PackageNotFoundError

import os

ARTIFACT_RELATIVE_DIR = Path("outputs/qwen3-0.6b-wikiner-lora-fp16")
project_roots = [Path.cwd(), *Path.cwd().parents]

configured_root = os.environ.get("WIKINER_PROJECT_ROOT")
if configured_root:
    project_roots.insert(0, Path(configured_root).expanduser())

# Common local workspace locations; these are optional fallbacks.
project_roots.extend([
    Path.home() / "Desktop" / "AI",
    Path.home() / "OneDrive" / "Desktop" / "AI",
])

# Prefer a root that already contains this experiment's artifacts.
artifact_root = next(
    (root for root in project_roots if (root / ARTIFACT_RELATIVE_DIR).exists()),
    None,
)

# If artifacts have not been created yet, use the notebook/repository root.
if artifact_root is None:
    artifact_root = next(
        (
            root for root in project_roots
            if (root / "2LoraProje" / "notebooks" / "02-turkish-wikiner-ner.ipynb").exists()
            or (root / "notebooks" / "02-turkish-wikiner-ner.ipynb").exists()
        ),
        Path.cwd(),
    )

ARTIFACT_DIR = artifact_root / ARTIFACT_RELATIVE_DIR
CHECKPOINT_DIR = ARTIFACT_DIR / "checkpoint-733"
BENCHMARK_DIR = ARTIFACT_DIR / "benchmark_checkpoint-733"

print("Python:", sys.version.split()[0])

packages = {
    "torch": "torch",
    "transformers": "transformers",
    "datasets": "datasets",
    "peft": "peft",
    "trl": "trl",
    "bitsandbytes": "bitsandbytes",
}

for module_name, package_name in packages.items():
    if importlib.util.find_spec(module_name) is None:
        print(f"{package_name}: not installed")
    else:
        try:
            print(f"{package_name}: {version(package_name)}")
        except PackageNotFoundError:
            print(f"{package_name}: installed (version unavailable)")

if importlib.util.find_spec("torch") is not None:
    import torch

    print("\nCUDA available:", torch.cuda.is_available())

    if torch.cuda.is_available():
        for i in range(torch.cuda.device_count()):
            gpu = torch.cuda.get_device_properties(i)
            memory_gb = gpu.total_memory / (1024 ** 3)
            print(f"GPU {i}: {gpu.name} — {memory_gb:.1f} GB memory")

Python: 3.10.20
torch: 2.6.0+cu124
transformers: 5.20.2
datasets: 5.1.0
peft: 0.21.2
trl: 1.14.2
bitsandbytes: 0.50.2

CUDA available: True
GPU 0: NVIDIA GeForce GTX 1650 Ti — 4.0 GB memory


## 8. LoRA and training configuration

The base model is loaded in FP16. LoRA uses rank 8, alpha 16, dropout 0.05, and attention/MLP projection modules. The per-device batch size is 2 with gradient accumulation of 8, giving an effective batch size of 16 on one GPU. The configured schedule is three epochs; the evaluated artifact is the saved checkpoint at step 733 (epoch 1).


In [ ]:
import torch
from transformers import AutoModelForCausalLM
from peft import LoraConfig, get_peft_model

model_name = "Qwen/Qwen3-0.6B"

if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU is not available.")

torch.cuda.empty_cache()

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.float16,
)

model.config.use_cache = False

lora_config = LoraConfig(
    task_type="CAUSAL_LM",
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    bias="none",
)

model = get_peft_model(model, lora_config)
model = model.to("cuda")

model.print_trainable_parameters()

torch.cuda.synchronize()
allocated_gb = torch.cuda.memory_allocated() / (1024 ** 3)
reserved_gb = torch.cuda.memory_reserved() / (1024 ** 3)

print("Model dtype:", next(model.parameters()).dtype)
print("Quantized:", getattr(model.get_base_model(), "is_quantized", False))
print(f"GPU memory allocated: {allocated_gb:.2f} GB")
print(f"GPU memory reserved: {reserved_gb:.2f} GB")

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

trainable params: 5,046,272 || all params: 601,096,192 || trainable%: 0.8395
Model dtype: torch.float16
Quantized: False
GPU memory allocated: 1.13 GB
GPU memory reserved: 1.15 GB


In [ ]:
from trl import SFTConfig, SFTTrainer

import math

train_batch_size = 2
eval_batch_size = 2
gradient_accumulation_steps = 8
num_train_epochs = 3

effective_batch_size = (
    train_batch_size
    * gradient_accumulation_steps
    * torch.cuda.device_count()
)
steps_per_epoch = math.ceil(len(sft_data["train"]) / effective_batch_size)
total_training_steps = steps_per_epoch * num_train_epochs
warmup_steps = round(total_training_steps * 0.03)

print("Steps per epoch:", steps_per_epoch)
print("Total training steps:", total_training_steps)
print("Warmup steps:", warmup_steps)

training_args = SFTConfig(
    output_dir=str(ARTIFACT_DIR),

    num_train_epochs=num_train_epochs,
    per_device_train_batch_size=train_batch_size,
    per_device_eval_batch_size=eval_batch_size,
    gradient_accumulation_steps=gradient_accumulation_steps,

    learning_rate=1e-4,
    lr_scheduler_type="cosine",
    warmup_steps=warmup_steps,

    fp16=True,
    bf16=False,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},

    max_length=256,
    assistant_only_loss=True,
    packing=False,
    eos_token=tokenizer.eos_token,

    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    logging_steps=25,
    optim="adamw_torch",
    dataloader_num_workers=0,
    report_to="none",
    seed=42,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=sft_data["train"],
    eval_dataset=sft_data["validation"],
    processing_class=tokenizer,
)

print("Train examples:", len(trainer.train_dataset))
print("Validation examples:", len(trainer.eval_dataset))
print("Prepared columns:", trainer.train_dataset.column_names)
print(
    "Effective batch size:",
    training_args.per_device_train_batch_size
    * training_args.gradient_accumulation_steps
    * torch.cuda.device_count(),
)

Steps per epoch: 733
Total training steps: 2199
Warmup steps: 66


Tokenizing train dataset:   0%|          | 0/11715 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/11715 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/11715 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/11715 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Train examples: 11714
Validation examples: 1000
Prepared columns: ['messages', 'input_ids', 'labels']
Effective batch size: 16


In [ ]:
from collections import Counter

def message_signature(example):
    return tuple(
        (message["role"], message["content"])
        for message in example["messages"]
    )

source_counts = Counter(
    message_signature(example)
    for example in sft_data["train"]
)

prepared_counts = Counter(
    message_signature(example)
    for example in trainer.train_dataset
)

dropped_signatures = list((source_counts - prepared_counts).elements())

print("Dropped examples:", len(dropped_signatures))

for example_number, signature in enumerate(dropped_signatures, start=1):
    print(f"\n--- Dropped example {example_number} ---")
    for role, content in signature:
        print(f"\n{role.upper()}:\n{content}")

## 9. Fine-tuning

Training is opt-in (`RUN_TRAINING = True`) because the configured three-epoch schedule is resource-intensive. The stored benchmark uses the selected `checkpoint-733` artifact from epoch 1; it does not claim that the full three-epoch schedule was completed.


In [ ]:
# Set RUN_TRAINING = True to start the configured three-epoch schedule.
RUN_TRAINING = False

if RUN_TRAINING:
    train_result = trainer.train()
else:
    print("Training skipped. Set RUN_TRAINING = True to start fine-tuning.")


## 10. Checkpoint spot check

In [ ]:
import random
import torch

checkpoint_dir = CHECKPOINT_DIR
adapter_name = "saved_checkpoint_733"

model = trainer.model

if adapter_name not in model.peft_config:
    model.load_adapter(
        str(checkpoint_dir),
        adapter_name=adapter_name,
        is_trainable=False,
    )

model.set_adapter(adapter_name, inference_mode=True)
model.eval()

validation_examples = []

for example in sft_data["validation"]:
    messages = example["messages"]
    assistant_message = messages[-1]

    if (
        assistant_message["role"] == "assistant"
        and assistant_message["content"].strip() != "Varlıklar:"
    ):
        validation_examples.append(messages)

random_generator = random.Random(42)
selected_examples = random_generator.sample(
    validation_examples,
    k=min(3, len(validation_examples)),
)

input_device = model.get_input_embeddings().weight.device
pad_token_id = (
    tokenizer.pad_token_id
    if tokenizer.pad_token_id is not None
    else tokenizer.eos_token_id
)

for example_number, full_messages in enumerate(selected_examples, start=1):
    prompt_messages = full_messages[:-1]
    expected_answer = full_messages[-1]["content"]

    encoded = tokenizer.apply_chat_template(
        prompt_messages,
        tokenize=True,
        add_generation_prompt=True,
        enable_thinking=False,
        return_dict=True,
        return_tensors="pt",
    )

    encoded = {
        key: value.to(input_device)
        for key, value in encoded.items()
    }

    with torch.inference_mode():
        generated = model.generate(
            **encoded,
            max_new_tokens=128,
            do_sample=False,
            pad_token_id=pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    prompt_length = encoded["input_ids"].shape[1]
    predicted_answer = tokenizer.decode(
        generated[0, prompt_length:],
        skip_special_tokens=True,
    ).strip()

    print(f"\n--- Test {example_number} ---")
    print("Girdi:", prompt_messages[-1]["content"])
    print("Beklenen:", expected_answer)
    print("Model:", predicted_answer)

## 11. Manual prompt examples

These hand-written examples are qualitative spot checks. Aggregate performance metrics come from the held-out test benchmark below.


In [ ]:
test_cases = [
    {
        "instruction": "Özellik çıkar.",
        "sentence": (
            "Orda Spike, First'ün etkisiyle Buffy'ye saldırır ve "
            "insanları öldürüp buraya gömdüğünü hatırlar."
        ),
        "expected": (
            "Varlıklar: PERSON: Spike; PERSON: First'ün; PERSON: Buffy'ye"
        ),
    },
    {
        "instruction": "Önemli bilgileri çıkar.",
        "sentence": (
            "Günümüzde Adana'da 514 okul öncesi eğitim kurumunda 19644 "
            "öğrenci ve 720 ilköğretim okulunda 324288 öğrenci eğitim görmektedir."
        ),
        "expected": "Varlıklar: GPE: Adana'da",
    },
    {
        "instruction": "Önemli kelimeleri çıkar.",
        "sentence": (
            "11 Temmuz 1927 tarihinde Filistin'de meydana gelen depremde "
            "önemli ölçüde hasar gören Kubbet üs Sahra, Ürdün, diğer Arap "
            "ülkeleri ve Türkiye'nin katkıları ile esaslı bir şekilde tamir edildi."
        ),
        "expected": (
            "Varlıklar: DATE: 11 Temmuz 1927 tarihinde; GPE: Filistin'de; "
            "EVENT: meydana gelen depremde; GPE: Ürdün; GPE: Arap ülkeleri; "
            "GPE: Türkiye'nin"
        ),
    },
    {
        "instruction": "Metindeki varlıkları bul.",
        "sentence": (
            "Beypazarı, Roma döneminde, İstanbul'u Ankara ve Bağdat'a "
            "bağlayan önemli büyük tarihi geçit yolları üzerinde bulunmaktaydı."
        ),
        "expected": (
            "Varlıklar: GPE: Beypazarı; GPE: Roma; GPE: İstanbul'u; "
            "GPE: Ankara; GPE: Bağdat'a"
        ),
    },
]

model.eval()

for example_number, test_case in enumerate(test_cases, start=1):
    messages = [
        {
            "role": "system",
            "content": (
                "Sen bir Named Entity Recognition (NER) modelisin. "
                "Verilen cümledeki varlıkları yalnızca PERSON, GPE, DATE, "
                "ORG, NORP, TITLE ve EVENT etiketlerini kullanarak çıkar. "
                'Yalnızca "Varlıklar: ETİKET: varlık; ..." biçiminde yanıt ver.'
            ),
        },
        {
            "role": "user",
            "content": (
                f"{test_case['instruction']}\n\n"
                f"Cümle: {test_case['sentence']}"
            ),
        },
    ]

    encoded = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        enable_thinking=False,
        return_dict=True,
        return_tensors="pt",
    )

    input_device = model.get_input_embeddings().weight.device
    encoded = {
        key: value.to(input_device)
        for key, value in encoded.items()
    }

    pad_token_id = (
        tokenizer.pad_token_id
        if tokenizer.pad_token_id is not None
        else tokenizer.eos_token_id
    )

    with torch.inference_mode():
        generated = model.generate(
            **encoded,
            max_new_tokens=128,
            do_sample=False,
            pad_token_id=pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    prompt_length = encoded["input_ids"].shape[1]
    predicted = tokenizer.decode(
        generated[0, prompt_length:],
        skip_special_tokens=True,
    ).strip()

    print(f"\n--- Test {example_number} ---")
    print("Cümle:", test_case["sentence"])
    print("Beklenen:", test_case["expected"])
    print("Model:", predicted)

## 12. Held-out test benchmark

The benchmark evaluates all 1,000 test examples in batches of 16. It records strict entity-and-label scores, per-label results, prompt-group results, output-format validity, empty-output behavior, throughput, and peak GPU memory. Predictions are saved immediately after inference for later analysis.


In [ ]:
import math
import time
import unicodedata
from collections import Counter, defaultdict

import pandas as pd
import torch
from IPython.display import display


BATCH_SIZE = 16
MAX_INPUT_TOKENS = 256
MAX_NEW_TOKENS = 128

ADAPTER_NAME = "saved_checkpoint_733"

TARGET_LABELS = [
    "PERSON",
    "GPE",
    "DATE",
    "ORG",
    "NORP",
    "TITLE",
    "EVENT",
]


# Activate the saved checkpoint adapter on the already loaded model.
model = trainer.model

if ADAPTER_NAME not in model.peft_config:
    model.load_adapter(
        str(CHECKPOINT_DIR),
        adapter_name=ADAPTER_NAME,
        is_trainable=False,
    )

model.set_adapter(ADAPTER_NAME, inference_mode=True)
model.eval()


# The raw test split provides gold BIO tags.
# The SFT test split provides the exact prompts and expected text outputs.
raw_test_data = training_data["test"]
test_chat_data = sft_data["test"]

if len(raw_test_data) != 1000 or len(test_chat_data) != 1000:
    raise ValueError(
        f"Expected 1,000 test examples; got "
        f"{len(raw_test_data)} raw and {len(test_chat_data)} chat examples."
    )


def normalize_entity_text(text):
    text = unicodedata.normalize("NFC", str(text))
    return " ".join(text.split())


test_records = []

for raw_example, chat_example in zip(raw_test_data, test_chat_data):
    messages = chat_example["messages"]
    user_content = messages[1]["content"]

    if "\n\nCümle:" in user_content:
        instruction = user_content.split("\n\nCümle:", 1)[0].strip()
    else:
        instruction = "No instruction"

    gold_entities = [
        (label, normalize_entity_text(entity_text))
        for label, entity_text in extract_entities(
            raw_example["tokens"],
            raw_example["tags"],
        )
    ]

    test_records.append({
        "prompt_messages": messages[:-1],
        "expected_answer": messages[-1]["content"],
        "gold_entities": gold_entities,
        "instruction": instruction,
        "sentence": detokenize(raw_example["tokens"]),
    })


def parse_entity_output(output_text):
    output_text = output_text.strip()

    if not output_text.startswith("Varlıklar:"):
        return [], False

    body = output_text[len("Varlıklar:"):].strip()

    if body == "Yok":
        return [], True

    if not body:
        return [], False

    parsed_entities = []
    valid_format = True

    for part in body.split(";"):
        part = part.strip()

        if ":" not in part:
            valid_format = False
            continue

        raw_label, raw_entity = part.split(":", 1)
        raw_label = raw_label.strip()
        entity_text = normalize_entity_text(raw_entity)
        canonical_label = raw_label.upper()

        if canonical_label not in TARGET_LABELS or not entity_text:
            valid_format = False
            continue

        if raw_label != canonical_label:
            valid_format = False

        parsed_entities.append((canonical_label, entity_text))

    return parsed_entities, valid_format


def compute_scores(true_positive, false_positive, false_negative):
    precision = (
        true_positive / (true_positive + false_positive)
        if true_positive + false_positive
        else 0.0
    )
    recall = (
        true_positive / (true_positive + false_negative)
        if true_positive + false_negative
        else 0.0
    )
    f1 = (
        2 * precision * recall / (precision + recall)
        if precision + recall
        else 0.0
    )

    return precision, recall, f1


def add_metrics(bucket, gold_counter, predicted_counter, valid_format):
    matches = gold_counter & predicted_counter

    bucket["examples"] += 1
    bucket["tp"] += sum(matches.values())
    bucket["fp"] += sum((predicted_counter - gold_counter).values())
    bucket["fn"] += sum((gold_counter - predicted_counter).values())
    bucket["valid_format"] += int(valid_format)

    entities_match = gold_counter == predicted_counter
    bucket["entity_exact"] += int(entities_match)
    bucket["strict_exact"] += int(entities_match and valid_format)

    if not gold_counter:
        bucket["gold_empty"] += 1
        bucket["correct_empty"] += int(
            not predicted_counter and valid_format
        )

    if not predicted_counter:
        bucket["predicted_empty"] += 1


def percentile(values, quantile):
    ordered_values = sorted(values)
    position = math.ceil(quantile * len(ordered_values)) - 1
    position = max(0, min(position, len(ordered_values) - 1))
    return ordered_values[position]


device = model.get_input_embeddings().weight.device
pad_token_id = (
    tokenizer.pad_token_id
    if tokenizer.pad_token_id is not None
    else tokenizer.eos_token_id
)

if pad_token_id is None:
    raise ValueError("Tokenizer needs a pad_token_id or eos_token_id.")

results = []
input_lengths = []
input_truncated_count = 0
output_limit_count = 0

if device.type == "cuda":
    torch.cuda.reset_peak_memory_stats(device)
    torch.cuda.synchronize(device)

start_time = time.perf_counter()

for batch_start in range(0, len(test_records), BATCH_SIZE):
    batch_records = test_records[batch_start:batch_start + BATCH_SIZE]
    batch_messages = [
        record["prompt_messages"]
        for record in batch_records
    ]

    # Tokenize each conversation using the tokenizer's chat template.
    tokenized_batch = tokenizer.apply_chat_template(
        batch_messages,
        tokenize=True,
        add_generation_prompt=True,
        enable_thinking=False,
        return_dict=True,
    )

    token_lists = tokenized_batch["input_ids"]
    input_lengths.extend(len(token_ids) for token_ids in token_lists)

    truncated_token_lists = []
    for token_ids in token_lists:
        if len(token_ids) > MAX_INPUT_TOKENS:
            input_truncated_count += 1
        truncated_token_lists.append(token_ids[:MAX_INPUT_TOKENS])

    # Left-pad manually so real EOS/end-of-turn tokens are not mistaken for padding.
    batch_width = max(len(token_ids) for token_ids in truncated_token_lists)

    batch_input_ids = torch.full(
        (len(batch_records), batch_width),
        fill_value=pad_token_id,
        dtype=torch.long,
    )
    batch_attention_mask = torch.zeros(
        (len(batch_records), batch_width),
        dtype=torch.long,
    )

    for row_index, token_ids in enumerate(truncated_token_lists):
        token_tensor = torch.tensor(token_ids, dtype=torch.long)
        batch_input_ids[row_index, -len(token_ids):] = token_tensor
        batch_attention_mask[row_index, -len(token_ids):] = 1

    batch_input_ids = batch_input_ids.to(device)
    batch_attention_mask = batch_attention_mask.to(device)

    with torch.inference_mode():
        generated_batch = model.generate(
            input_ids=batch_input_ids,
            attention_mask=batch_attention_mask,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    for row_index, record in enumerate(batch_records):
        generated_ids = generated_batch[row_index, batch_width:].tolist()

        if tokenizer.eos_token_id in generated_ids:
            eos_position = generated_ids.index(tokenizer.eos_token_id)
            generated_ids = generated_ids[:eos_position]
        else:
            output_limit_count += 1

        predicted_text = tokenizer.decode(
            generated_ids,
            skip_special_tokens=True,
        ).strip()

        predicted_entities, valid_format = parse_entity_output(predicted_text)

        results.append({
            **record,
            "predicted_answer": predicted_text,
            "predicted_entities": predicted_entities,
            "valid_format": valid_format,
        })

if device.type == "cuda":
    torch.cuda.synchronize(device)

elapsed_seconds = time.perf_counter() - start_time


# Calculate overall and per-instruction entity metrics.
overall = Counter()
per_label = {label: Counter() for label in TARGET_LABELS}
per_instruction = defaultdict(Counter)

for result in results:
    gold_counter = Counter(result["gold_entities"])
    predicted_counter = Counter(result["predicted_entities"])

    add_metrics(
        overall,
        gold_counter,
        predicted_counter,
        result["valid_format"],
    )
    add_metrics(
        per_instruction[result["instruction"]],
        gold_counter,
        predicted_counter,
        result["valid_format"],
    )

    matches = gold_counter & predicted_counter

    for (label, _), count in matches.items():
        per_label[label]["tp"] += count

    for (label, _), count in (predicted_counter - gold_counter).items():
        per_label[label]["fp"] += count

    for (label, _), count in (gold_counter - predicted_counter).items():
        per_label[label]["fn"] += count

    for (label, _), count in gold_counter.items():
        per_label[label]["support"] += count

    for (label, _), count in predicted_counter.items():
        per_label[label]["predicted"] += count


micro_precision, micro_recall, micro_f1 = compute_scores(
    overall["tp"],
    overall["fp"],
    overall["fn"],
)

supported_label_f1 = []

for label in TARGET_LABELS:
    label_scores = compute_scores(
        per_label[label]["tp"],
        per_label[label]["fp"],
        per_label[label]["fn"],
    )
    if per_label[label]["support"] > 0:
        supported_label_f1.append(label_scores[2])

macro_f1 = (
    sum(supported_label_f1) / len(supported_label_f1)
    if supported_label_f1
    else 0.0
)

cuda_peak_gb = (
    torch.cuda.max_memory_allocated(device) / (1024 ** 3)
    if device.type == "cuda"
    else float("nan")
)

summary_rows = [
    ("Test examples", len(results)),
    ("Batch size", BATCH_SIZE),
    ("Batches", math.ceil(len(results) / BATCH_SIZE)),
    ("Entity micro precision (%)", round(100 * micro_precision, 2)),
    ("Entity micro recall (%)", round(100 * micro_recall, 2)),
    ("Entity micro F1 (%)", round(100 * micro_f1, 2)),
    ("Macro F1 across supported labels (%)", round(100 * macro_f1, 2)),
    (
        "Entity exact match per sentence (%)",
        round(100 * overall["entity_exact"] / len(results), 2),
    ),
    (
        "Strict exact match incl. valid format (%)",
        round(100 * overall["strict_exact"] / len(results), 2),
    ),
    (
        "Valid output format (%)",
        round(100 * overall["valid_format"] / len(results), 2),
    ),
    ("Gold empty sentences", overall["gold_empty"]),
    (
        "Correct empty outputs among gold-empty (%)",
        round(
            100 * overall["correct_empty"] / overall["gold_empty"],
            2,
        ) if overall["gold_empty"] else 0.0,
    ),
    (
        "Prompts over input limit",
        input_truncated_count,
    ),
    (
        "Outputs reaching max_new_tokens",
        output_limit_count,
    ),
    (
        "Mean input tokens",
        round(sum(input_lengths) / len(input_lengths), 1),
    ),
    ("P95 input tokens", percentile(input_lengths, 0.95)),
    ("Max input tokens", max(input_lengths)),
    ("Inference time, seconds", round(elapsed_seconds, 2)),
    (
        "Average latency, ms/example",
        round(1000 * elapsed_seconds / len(results), 2),
    ),
    (
        "Throughput, examples/second",
        round(len(results) / elapsed_seconds, 2),
    ),
    ("Peak GPU allocated, GB", round(cuda_peak_gb, 2)),
]

print("=== OVERALL BENCHMARK ===")
display(pd.DataFrame(summary_rows, columns=["Metric", "Value"]))


label_rows = []

for label in TARGET_LABELS:
    scores = compute_scores(
        per_label[label]["tp"],
        per_label[label]["fp"],
        per_label[label]["fn"],
    )

    label_rows.append({
        "Label": label,
        "Gold support": per_label[label]["support"],
        "Predicted": per_label[label]["predicted"],
        "TP": per_label[label]["tp"],
        "FP": per_label[label]["fp"],
        "FN": per_label[label]["fn"],
        "Precision (%)": round(100 * scores[0], 2),
        "Recall (%)": round(100 * scores[1], 2),
        "F1 (%)": round(100 * scores[2], 2),
    })

print("\n=== PER-LABEL ENTITY SCORES ===")
display(pd.DataFrame(label_rows))


instruction_rows = []

for instruction, stats in sorted(per_instruction.items()):
    scores = compute_scores(stats["tp"], stats["fp"], stats["fn"])

    instruction_rows.append({
        "Instruction": instruction,
        "Examples": stats["examples"],
        "Micro F1 (%)": round(100 * scores[2], 2),
        "Entity exact match (%)": round(
            100 * stats["entity_exact"] / stats["examples"],
            2,
        ),
        "Valid format (%)": round(
            100 * stats["valid_format"] / stats["examples"],
            2,
        ),
    })

print("\n=== RESULTS BY INSTRUCTION ===")
display(pd.DataFrame(instruction_rows))


mismatches = [
    result
    for result in results
    if (
        Counter(result["gold_entities"])
        != Counter(result["predicted_entities"])
        or not result["valid_format"]
    )
]

print(f"\n=== FIRST 5 MISMATCHES ({len(mismatches)} total) ===")

for result in mismatches[:5]:
    print(f"\nInstruction: {result['instruction']}")
    print(f"Sentence: {result['sentence']}")
    print(f"Gold: {result['expected_answer']}")
    print(f"Model: {result['predicted_answer']}")
    print(f"Valid format: {result['valid_format']}")
    

=== OVERALL BENCHMARK ===


,Metric,Value
0,Test examples,1000.00
1,Batch size,16.00
2,Batches,63.00
3,Entity micro precision (%),73.46
4,Entity micro recall (%),76.46
5,Entity micro F1 (%),74.93
6,Macro F1 across supported labels (%),67.14
7,Entity exact match per sentence (%),52.90
8,Strict exact match incl. valid format (%),52.90
9,Valid output format (%),100.00



=== PER-LABEL ENTITY SCORES ===


,Label,Gold support,Predicted,TP,FP,FN,Precision (%),Recall (%),F1 (%)
0,PERSON,654,716,575,141,79,80.31,87.92,83.94
1,GPE,566,636,462,174,104,72.64,81.63,76.87
2,DATE,343,382,288,94,55,75.39,83.97,79.45
3,ORG,250,242,167,75,83,69.01,66.80,67.89
4,NORP,212,232,155,77,57,66.81,73.11,69.82
5,TITLE,153,68,36,32,117,52.94,23.53,32.58
6,EVENT,95,90,55,35,40,61.11,57.89,59.46



=== RESULTS BY INSTRUCTION ===


,Instruction,Examples,Micro F1 (%),Entity exact match (%),Valid format (%)
0,Anahtar kelimeleri çıkar.,80,73.22,47.50,100.0
1,Cümlede geçen hedef varlıkları belirle.,80,76.88,57.50,100.0
2,"Kişi, yer, tarih, kurum, grup, unvan ve olay a...",80,77.99,56.25,100.0
3,Metindeki varlıkları bul.,80,70.91,51.25,100.0
4,No instruction,200,75.45,51.00,100.0
5,Tanımlı etiketlerdeki varlıkları çıkar.,80,71.20,42.50,100.0
6,Varlıkları belirtilen kategorilere göre sınıfl...,80,73.54,63.75,100.0
7,Varlıkları türleriyle listele.,80,75.44,41.25,100.0
8,Önemli bilgileri çıkar.,80,71.04,53.75,100.0
9,Önemli kelimeleri çıkar.,80,78.39,67.50,100.0



=== FIRST 5 MISMATCHES (471 total) ===

Instruction: Tanımlı etiketlerdeki varlıkları çıkar.
Sentence: İsviçre Federal Sarayı İsviçre Federal Meclisi ve İsviçre Federal Konseyine ev sahipliği yapan Bern'de bulunan tarihi yapı.
Gold: Varlıklar: ORG: İsviçre Federal Meclisi; ORG: İsviçre Federal Konseyine; GPE: Bern'de
Model: Varlıklar: GPE: İsviçre; ORG: İsviçre Federal Meclisi; ORG: İsviçre Federal Konseyine; GPE: Bern'de
Valid format: True

Instruction: Varlıkları türleriyle listele.
Sentence: Kral Creon gelir ve Medea'nın Korint şehrinden ayrılmadığını görüp çok kızar ve muhafız askerlerine onun tutup şehir dışına atmalarına emir verir.
Gold: Varlıklar: PERSON: Kral Creon; PERSON: Medea'nın; GPE: Korint
Model: Varlıklar: TITLE: Kral; PERSON: Creon; PERSON: Medea'nın; GPE: Korint
Valid format: True

Instruction: Kişi, yer, tarih, kurum, grup, unvan ve olay adlarını tespit et.
Sentence: 1995: Posta. 1995: Fanatik. 1996: Radikal. 1996 2007: Gözcü. 2002 2011: Vatan (Yıldırım Demirören'e

In [ ]:
import json
from pathlib import Path

output_dir = BENCHMARK_DIR
output_dir.mkdir(parents=True, exist_ok=True)

jsonl_path = output_dir / "test_predictions.jsonl"
csv_path = output_dir / "test_predictions.csv"

jsonl_records = []
csv_records = []

for example_id, result in enumerate(results):
    record = {
        "example_id": example_id,
        "checkpoint": "checkpoint-733",
        "instruction": result["instruction"],
        "sentence": result["sentence"],
        "expected_answer": result["expected_answer"],
        "predicted_answer": result["predicted_answer"],
        "gold_entities": [
            {"label": label, "text": text}
            for label, text in result["gold_entities"]
        ],
        "predicted_entities": [
            {"label": label, "text": text}
            for label, text in result["predicted_entities"]
        ],
        "valid_format": result["valid_format"],
    }
    jsonl_records.append(record)

    csv_records.append({
        **record,
        "gold_entities": json.dumps(record["gold_entities"], ensure_ascii=False),
        "predicted_entities": json.dumps(
            record["predicted_entities"],
            ensure_ascii=False,
        ),
    })

with jsonl_path.open("w", encoding="utf-8") as file:
    for record in jsonl_records:
        file.write(json.dumps(record, ensure_ascii=False) + "\n")

pd.DataFrame(csv_records).to_csv(csv_path, index=False, encoding="utf-8-sig")

print(f"Saved {len(results)} predictions:")
print(jsonl_path.resolve())
print(csv_path.resolve())

Saved 1000 predictions:
<project-root>/outputs\qwen3-0.6b-wikiner-lora-fp16\benchmark_checkpoint-733\test_predictions.jsonl
<project-root>/outputs\qwen3-0.6b-wikiner-lora-fp16\benchmark_checkpoint-733\test_predictions.csv


## 13. Detailed metrics and error analysis

In [3]:
import json
import os
import random
import re
import unicodedata
from collections import Counter, defaultdict
from pathlib import Path

import pandas as pd
from IPython.display import display


artifact_relative_dir = Path("outputs/qwen3-0.6b-wikiner-lora-fp16")
relative_prediction_path = (
    artifact_relative_dir
    / "benchmark_checkpoint-733"
    / "test_predictions.jsonl"
)

search_roots = [Path.cwd(), *Path.cwd().parents]

configured_root = os.environ.get("WIKINER_PROJECT_ROOT")
if configured_root:
    search_roots.insert(0, Path(configured_root).expanduser())

# Fallbacks for common local workspace locations.
search_roots.extend([
    Path.home() / "Desktop" / "AI",
    Path.home() / "OneDrive" / "Desktop" / "AI",
])
search_roots = list(dict.fromkeys(root.resolve() for root in search_roots))

prediction_path = next(
    (
        root / relative_prediction_path
        for root in search_roots
        if (root / relative_prediction_path).is_file()
    ),
    None,
)

if prediction_path is None:
    searched_paths = [
        str(root / relative_prediction_path)
        for root in search_roots
    ]
    raise FileNotFoundError(
        "Could not find saved test_predictions.jsonl. Searched these locations:\n"
        + "\n".join(searched_paths)
    )

print("Using predictions:", prediction_path)

with prediction_path.open("r", encoding="utf-8") as file:
    predictions = [
        json.loads(line)
        for line in file
        if line.strip()
    ]

if not predictions:
    raise ValueError(f"No predictions found in: {prediction_path}")

word_pattern = re.compile(
    r"[^\W_]+(?:['’][^\W_]+)*",
    flags=re.UNICODE,
)

turkish_case_translation = str.maketrans({
    "I": "ı",
    "İ": "i",
})


def normalize_surface(text):
    text = unicodedata.normalize("NFC", str(text))
    return " ".join(text.split())


def normalize_word(word):
    word = unicodedata.normalize("NFC", str(word))
    return word.translate(turkish_case_translation).lower()


def extract_words(text):
    normalized_text = unicodedata.normalize("NFC", str(text))
    return [
        normalize_word(word)
        for word in word_pattern.findall(normalized_text)
    ]


def add_overlap_counts(bucket, prefix, gold_counter, predicted_counter):
    matched = gold_counter & predicted_counter

    bucket[f"{prefix}_tp"] += sum(matched.values())
    bucket[f"{prefix}_fp"] += sum(
        (predicted_counter - gold_counter).values()
    )
    bucket[f"{prefix}_fn"] += sum(
        (gold_counter - predicted_counter).values()
    )


def get_scores(bucket, prefix):
    true_positive = bucket[f"{prefix}_tp"]
    false_positive = bucket[f"{prefix}_fp"]
    false_negative = bucket[f"{prefix}_fn"]

    precision = (
        true_positive / (true_positive + false_positive)
        if true_positive + false_positive
        else 0.0
    )
    recall = (
        true_positive / (true_positive + false_negative)
        if true_positive + false_negative
        else 0.0
    )
    f1 = (
        2 * precision * recall / (precision + recall)
        if precision + recall
        else 0.0
    )

    return precision, recall, f1


overall = Counter()
instruction_stats = defaultdict(Counter)

target_labels = [
    "PERSON",
    "GPE",
    "DATE",
    "ORG",
    "NORP",
    "TITLE",
    "EVENT",
]

per_label_stats = {
    label: Counter()
    for label in target_labels
}

# Translate instruction names for display only.
# The original Turkish instructions remain unchanged in the prediction data.
instruction_display_names = {
    "Anahtar kelimeleri çıkar.": "Extract keywords.",
    "Cümlede geçen hedef varlıkları belirle.": (
        "Identify the target entities in the sentence."
    ),
    "Kişi, yer, tarih, kurum, grup, unvan ve olay adlarını tespit et.": (
        "Detect person, place, date, organization, group, title, and event names."
    ),
    "Metindeki varlıkları bul.": "Find the entities in the text.",
    "Tanımlı etiketlerdeki varlıkları çıkar.": (
        "Extract entities with the defined labels."
    ),
    "Varlıkları belirtilen kategorilere göre sınıflandır.": (
        "Classify entities by the specified categories."
    ),
    "Varlıkları türleriyle listele.": "List entities with their types.",
    "Önemli bilgileri çıkar.": "Extract important information.",
    "Önemli kelimeleri çıkar.": "Extract important words.",
    "Özellik çıkar.": "Extract features.",
    "No instruction": "No instruction",
}


for row in predictions:
    gold_entities = row["gold_entities"]
    predicted_entities = row["predicted_entities"]

    gold_word_occurrences = Counter(
        word
        for entity in gold_entities
        for word in extract_words(entity["text"])
    )
    predicted_word_occurrences = Counter(
        word
        for entity in predicted_entities
        for word in extract_words(entity["text"])
    )

    # Count each distinct word once per sentence.
    gold_word_set = Counter({
        word: 1 for word in gold_word_occurrences
    })
    predicted_word_set = Counter({
        word: 1 for word in predicted_word_occurrences
    })

    gold_surface_mentions = Counter(
        normalize_surface(entity["text"])
        for entity in gold_entities
    )
    predicted_surface_mentions = Counter(
        normalize_surface(entity["text"])
        for entity in predicted_entities
    )

    gold_typed_mentions = Counter(
        (
            str(entity["label"]).strip().upper(),
            normalize_surface(entity["text"]),
        )
        for entity in gold_entities
    )
    predicted_typed_mentions = Counter(
        (
            str(entity["label"]).strip().upper(),
            normalize_surface(entity["text"]),
        )
        for entity in predicted_entities
    )

    instruction = row.get("instruction", "No instruction")
    scopes = [overall, instruction_stats[instruction]]

    for scope in scopes:
        scope["examples"] += 1

        add_overlap_counts(
            scope,
            "word_set",
            gold_word_set,
            predicted_word_set,
        )
        add_overlap_counts(
            scope,
            "word_occurrence",
            gold_word_occurrences,
            predicted_word_occurrences,
        )
        add_overlap_counts(
            scope,
            "surface_mention",
            gold_surface_mentions,
            predicted_surface_mentions,
        )
        add_overlap_counts(
            scope,
            "typed_mention",
            gold_typed_mentions,
            predicted_typed_mentions,
        )

        scope["word_set_exact"] += int(
            gold_word_set == predicted_word_set
        )
        scope["surface_mention_exact"] += int(
            gold_surface_mentions == predicted_surface_mentions
        )
        scope["typed_mention_exact"] += int(
            gold_typed_mentions == predicted_typed_mentions
        )
        scope["valid_format"] += int(bool(row["valid_format"]))

        matched_surface_mentions = (
            gold_surface_mentions & predicted_surface_mentions
        )
        correctly_labeled_matches = (
            gold_typed_mentions & predicted_typed_mentions
        )

        scope["matched_surface_mentions"] += sum(
            matched_surface_mentions.values()
        )
        scope["correct_label_matches"] += sum(
            correctly_labeled_matches.values()
        )

        if not gold_entities:
            scope["gold_empty_sentences"] += 1
            scope["correct_empty_outputs"] += int(
                not predicted_entities and bool(row["valid_format"])
            )

    for label in target_labels:
        gold_for_label = Counter({
            text: count
            for (entity_label, text), count in gold_typed_mentions.items()
            if entity_label == label
        })
        predicted_for_label = Counter({
            text: count
            for (entity_label, text), count in predicted_typed_mentions.items()
            if entity_label == label
        })

        add_overlap_counts(
            per_label_stats[label],
            "typed",
            gold_for_label,
            predicted_for_label,
        )
        per_label_stats[label]["gold_support"] += sum(
            gold_for_label.values()
        )
        per_label_stats[label]["predicted_count"] += sum(
            predicted_for_label.values()
        )


metric_definitions = [
    ("Unique entity words, per sentence", "word_set"),
    ("Entity word occurrences, per sentence", "word_occurrence"),
    ("Exact entity phrases, ignoring labels", "surface_mention"),
    ("Exact entity phrases, including labels", "typed_mention"),
]

summary_rows = []

for metric_name, prefix in metric_definitions:
    precision, recall, f1 = get_scores(overall, prefix)

    summary_rows.append({
        "Metric": metric_name,
        "TP": overall[f"{prefix}_tp"],
        "FP": overall[f"{prefix}_fp"],
        "FN": overall[f"{prefix}_fn"],
        "Precision (%)": round(100 * precision, 2),
        "Recall (%)": round(100 * recall, 2),
        "F1 (%)": round(100 * f1, 2),
    })

print(f"Examples loaded: {len(predictions)}")
print("\n=== OVERALL METRICS ===")
display(pd.DataFrame(summary_rows))


exact_match_rows = [
    {
        "Metric": "Exact unique-word set per sentence (%)",
        "Value": round(
            100 * overall["word_set_exact"] / overall["examples"],
            2,
        ),
    },
    {
        "Metric": "Exact phrase set per sentence, labels ignored (%)",
        "Value": round(
            100 * overall["surface_mention_exact"] / overall["examples"],
            2,
        ),
    },
    {
        "Metric": "Exact phrase set per sentence, labels included (%)",
        "Value": round(
            100 * overall["typed_mention_exact"] / overall["examples"],
            2,
        ),
    },
    {
        "Metric": "Label agreement among exact phrase matches (%)",
        "Value": round(
            100
            * overall["correct_label_matches"]
            / overall["matched_surface_mentions"],
            2,
        ) if overall["matched_surface_mentions"] else 0.0,
    },
    {
        "Metric": "Valid output format (%)",
        "Value": round(
            100 * overall["valid_format"] / overall["examples"],
            2,
        ),
    },
    {
        "Metric": "Gold-empty sentences",
        "Value": overall["gold_empty_sentences"],
    },
    {
        "Metric": "Correct empty outputs among gold-empty (%)",
        "Value": round(
            100
            * overall["correct_empty_outputs"]
            / overall["gold_empty_sentences"],
            2,
        ) if overall["gold_empty_sentences"] else 0.0,
    },
]

print("\n=== SENTENCE-LEVEL AND FORMAT METRICS ===")
display(pd.DataFrame(exact_match_rows))


instruction_rows = []

for instruction, stats in sorted(instruction_stats.items()):
    word_precision, word_recall, word_f1 = get_scores(
        stats,
        "word_set",
    )
    phrase_precision, phrase_recall, phrase_f1 = get_scores(
        stats,
        "surface_mention",
    )
    typed_precision, typed_recall, typed_f1 = get_scores(
        stats,
        "typed_mention",
    )

    instruction_rows.append({
        "Instruction": instruction_display_names.get(instruction, instruction),
        "Examples": stats["examples"],
        "Word-set Precision (%)": round(100 * word_precision, 2),
        "Word-set Recall (%)": round(100 * word_recall, 2),
        "Word-set F1 (%)": round(100 * word_f1, 2),
        "Word-set exact sentence (%)": round(
            100 * stats["word_set_exact"] / stats["examples"],
            2,
        ),
        "Label-free phrase F1 (%)": round(100 * phrase_f1, 2),
        "Typed phrase F1 (%)": round(100 * typed_f1, 2),
    })

print("\n=== METRICS BY INSTRUCTION ===")
display(pd.DataFrame(instruction_rows))


label_rows = []

for label in target_labels:
    stats = per_label_stats[label]
    precision, recall, f1 = get_scores(stats, "typed")

    label_rows.append({
        "Label": label,
        "Gold support": stats["gold_support"],
        "Predicted": stats["predicted_count"],
        "TP": stats["typed_tp"],
        "FP": stats["typed_fp"],
        "FN": stats["typed_fn"],
        "Precision (%)": round(100 * precision, 2),
        "Recall (%)": round(100 * recall, 2),
        "F1 (%)": round(100 * f1, 2),
    })

print("\n=== LABEL-SPECIFIC SCORES ===")
display(pd.DataFrame(label_rows))


# Show a compact sample of strict entity-level mismatches.
entity_mismatches = []

for row in predictions:
    gold_typed_entities = Counter(
        (entity["label"].strip().upper(), normalize_surface(entity["text"]))
        for entity in row["gold_entities"]
    )
    predicted_typed_entities = Counter(
        (entity["label"].strip().upper(), normalize_surface(entity["text"]))
        for entity in row["predicted_entities"]
    )

    if gold_typed_entities != predicted_typed_entities:
        entity_mismatches.append(row)

example_rng = random.Random(42)
example_rows = example_rng.sample(
    entity_mismatches,
    k=min(12, len(entity_mismatches)),
)

print(
    f"\n=== ENTITY EXAMPLES "
    f"({len(example_rows)} of {len(entity_mismatches)} mismatches) ==="
)

for row in example_rows:
    gold_text = "; ".join(
        f"{entity['label']}: {entity['text']}"
        for entity in row["gold_entities"]
    ) or "None"
    predicted_text = "; ".join(
        f"{entity['label']}: {entity['text']}"
        for entity in row["predicted_entities"]
    ) or "None"

    print(f"\nSentence: {row['sentence']}")
    print(f"Gold entities: {gold_text}")
    print(f"Predicted entities: {predicted_text}")

Using predictions: <project-root>/outputs\qwen3-0.6b-wikiner-lora-fp16\benchmark_checkpoint-733\test_predictions.jsonl
Examples loaded: 1000

=== OVERALL METRICS ===


,Metric,TP,FP,FN,Precision (%),Recall (%),F1 (%)
0,"Unique entity words, per sentence",3478,441,371,88.75,90.36,89.55
1,"Entity word occurrences, per sentence",3591,466,387,88.51,90.27,89.38
2,"Exact entity phrases, ignoring labels",1833,533,440,77.47,80.64,79.03
3,"Exact entity phrases, including labels",1738,628,535,73.46,76.46,74.93



=== SENTENCE-LEVEL AND FORMAT METRICS ===


,Metric,Value
0,Exact unique-word set per sentence (%),62.50
1,"Exact phrase set per sentence, labels ignored (%)",57.20
2,"Exact phrase set per sentence, labels included...",52.90
3,Label agreement among exact phrase matches (%),94.82
4,Valid output format (%),100.00
5,Gold-empty sentences,187.00
6,Correct empty outputs among gold-empty (%),78.07



=== METRICS BY INSTRUCTION ===


,Instruction,Examples,Word-set Precision (%),Word-set Recall (%),Word-set F1 (%),Word-set exact sentence (%),Label-free phrase F1 (%),Typed phrase F1 (%)
0,Extract keywords.,80,91.55,91.55,91.55,62.50,77.64,73.22
1,Identify the target entities in the sentence.,80,91.96,90.38,91.16,67.50,82.26,76.88
2,"Detect person, place, date, organization, grou...",80,94.41,92.35,93.37,65.00,81.82,77.99
3,Find the entities in the text.,80,85.46,85.16,85.31,57.50,73.33,70.91
4,No instruction,200,90.75,90.86,90.80,63.00,80.51,75.45
5,Extract entities with the defined labels.,80,83.89,92.64,88.05,57.50,76.96,71.20
6,Classify entities by the specified categories.,80,81.22,91.71,86.15,72.50,78.35,73.54
7,List entities with their types.,80,86.31,84.69,85.49,48.75,77.97,75.44
8,Extract important information.,80,89.03,89.59,89.31,60.00,76.50,71.04
9,Extract important words.,80,93.84,89.54,91.64,70.00,80.12,78.39



=== LABEL-SPECIFIC SCORES ===


,Label,Gold support,Predicted,TP,FP,FN,Precision (%),Recall (%),F1 (%)
0,PERSON,654,716,575,141,79,80.31,87.92,83.94
1,GPE,566,636,462,174,104,72.64,81.63,76.87
2,DATE,343,382,288,94,55,75.39,83.97,79.45
3,ORG,250,242,167,75,83,69.01,66.80,67.89
4,NORP,212,232,155,77,57,66.81,73.11,69.82
5,TITLE,153,68,36,32,117,52.94,23.53,32.58
6,EVENT,95,90,55,35,40,61.11,57.89,59.46



=== ENTITY EXAMPLES (12 of 471 mismatches) ===

Sentence: Bu ilk kez bir ABD takımının bu yarışmada altın madalya almasıydı .. Vancouver, Kanada'da düzenlenen 2010 Kış Olimpiyatları'nda çift buz dansı kategorisimde Canada'dan Virtue & Moir'in ardında ikinci sırada yer aldı ve gümüş madalya kazandı.
Gold entities: GPE: ABD; GPE: Vancouver, Kanada'da; EVENT: 2010 Kış Olimpiyatları'nda; GPE: Canada'dan; PERSON: Virtue; PERSON: Moir'in
Predicted entities: ORG: ABD; GPE: Vancouver; GPE: Kanada'da; EVENT: 2010 Kış Olimpiyatları'nda; GPE: Canada'dan; PERSON: Virtue & Moir'in

Sentence: 2002 2003 sezonundan 2004 2005 sezonuna kadar, Türkiye 2. Ligi B Grubu'nda kategoriler A, B ve C diye oynanıyordu.
Gold entities: DATE: 2002 2003; DATE: 2004 2005; ORG: Türkiye 2. Ligi B Grubu'nda
Predicted entities: DATE: 2002 2003; DATE: 2004 2005; GPE: Türkiye; EVENT: 2. Ligi B Grubu'nda

Sentence: Amerika Birleşik Devletleri'nde feminizm akımının yenilenmesine büyük katkısı bulunan Kathleen Hanna, aynı zam

### Performance interpretation

The selected checkpoint provides a promising baseline. It reaches **89.55% F1** on the custom unique-entity-word overlap metric, showing strong coverage of entity-related words. On the stricter exact-phrase-and-label metric, it reaches **74.93% micro F1** and **67.14% macro F1**; **52.90%** of sentences have an exact typed-entity set match. Output formatting is valid for **100%** of test examples.

PERSON (**83.94% F1**), DATE (**79.45%**), and GPE (**76.87%**) are the strongest label results. TITLE is the clearest next improvement opportunity (**32.58% F1**). Per-instruction results are useful for exploration; each explicit instruction has 80 test examples, so small differences are preliminary.

The word-overlap metric is more permissive than exact NER scoring because it ignores labels and entity boundaries. It complements the strict phrase-and-label results rather than replacing them.
